In [10]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('POSTGRES_USER', 'postgres')}:{os.getenv('POSTGRES_PASSWORD', 'postgres')}"
    f"@{os.getenv('POSTGRES_HOST', 'localhost')}:{os.getenv('POSTGRES_PORT', '5434')}/{os.getenv('POSTGRES_DB', 'cofair_db')}"
)

In [11]:
# Connect to local Docker Postgres (ensure docker-compose is running)
pricing = pd.read_sql("SELECT * FROM staging.stg_pricing", engine)

In [22]:
# IMPORTANT: The usage table is very big, so please only select a small sample of rows
pd.read_sql("SELECT COUNT(*) FROM staging.stg_usage WHERE session_id IS NOT NULL", engine)

,count
0,6557


In [23]:
usage = pd.read_sql("SELECT * FROM staging.stg_usage WHERE session_id IS NOT NULL LIMIT 3000", engine)

## Feel free to play around with the data here!

Data is loaded from your local Postgres DB (`staging.stg_pricing`).

In [9]:
print("Pricing Rows:",len(pricing))
pricing.head(5)

Pricing Rows: 351


,content_sha256,ingested_at,pricing_version,currency,provider,model,sku_type,price_per_1m_tokens_usd,price_per_1k_tokens_usd
0,eeb315ea05288376587aa3ceff42f44505b6c6e555fcfa...,2026-03-20 05:22:34.596942+00:00,2026-03-11T19:39:39Z,USD,anthropic,claude-opus-4.6,base_input_tokens,5.0,0.005
1,eeb315ea05288376587aa3ceff42f44505b6c6e555fcfa...,2026-03-20 05:22:34.596942+00:00,2026-03-11T19:39:39Z,USD,anthropic,claude-opus-4.5,base_input_tokens,5.0,0.005
2,eeb315ea05288376587aa3ceff42f44505b6c6e555fcfa...,2026-03-20 05:22:34.596942+00:00,2026-03-11T19:39:39Z,USD,anthropic,claude-opus-4.1,base_input_tokens,15.0,0.015
3,eeb315ea05288376587aa3ceff42f44505b6c6e555fcfa...,2026-03-20 05:22:34.596942+00:00,2026-03-11T19:39:39Z,USD,anthropic,claude-opus-4,base_input_tokens,15.0,0.015
4,eeb315ea05288376587aa3ceff42f44505b6c6e555fcfa...,2026-03-20 05:22:34.596942+00:00,2026-03-11T19:39:39Z,USD,anthropic,claude-sonnet-4.6,base_input_tokens,3.0,0.003


In [ ]:
# Avg. Pricing by Provider
pricing.groupby("provider").agg({"price_per_1m_tokens_usd":"mean","price_per_1k_tokens_usd":"mean"})

,price_per_1m_tokens_usd,price_per_1k_tokens_usd
provider,,
anthropic,17.262500,0.017263
google,7.448028,0.007448
openai,7.394021,0.007394


In [24]:
usage

,dataset_id,session_id,model,git_branch,start_time,end_time,project,messages,input_tokens,output_tokens
0,peteromallet/dataclaw-peteromallet,d3a1a2cb-16ae-4cf1-8b80-23e528d64ad9,claude-opus-4-6,main,2026-02-12 17:35:03.944,2026-02-18 12:20:12.774,Arnold,"[{'role': 'user', 'content': 'can you run this...",1490757,795
1,peteromallet/dataclaw-peteromallet,2ca7768e-d794-4eaa-bec3-cff949b9df96,claude-haiku-4-5-20251001,main,2026-02-24 12:31:08.924,2026-02-24 12:55:11.830,Headless-WGP-Orchestrator,"[{'role': 'user', 'content': 'Can you try to u...",5713237,3162
2,peteromallet/dataclaw-peteromallet,078446fe-c679-4abd-8be1-0e4cc935892a,claude-opus-4-6,main,2026-02-12 22:19:08.988,2026-02-12 22:27:54.566,Headless-Wan2GP,"[{'role': 'user', 'content': 'Hey can you see ...",981855,434
3,peteromallet/dataclaw-peteromallet,104e116b-a969-4548-8b1c-bc59145b3d48,claude-opus-4-6,main,2026-02-13 12:13:11.250,2026-02-13 12:35:46.718,Headless-Wan2GP,"[{'role': 'user', 'content': 'can you run the ...",10907618,11481
4,peteromallet/dataclaw-peteromallet,20e6c204-3714-4204-a24d-66af004e2f57,claude-haiku-4-5-20251001,codex/pr19-layered-port,2026-02-24 11:51:28.696,2026-02-24 14:41:46.533,Headless-Wan2GP,"[{'role': 'user', 'content': 'Can you check th...",49644770,25577
...,...,...,...,...,...,...,...,...,...,...
2995,hitlabstudios/dataclaw-peteromallet,a29a9b6d-d706-459e-85e4-763d642a98ce,claude-opus-4-5-20251101,,2026-01-30 09:39:40.504,2026-01-30 10:29:11.778,~home,"[{'role': 'user', 'content': 'See on batch edi...",36672499,1157
2996,hitlabstudios/dataclaw-peteromallet,ebc3d989-6bda-4611-8f0e-890c53ea3129,claude-sonnet-4-5-20250929,main,2026-01-20 19:20:08.065,2026-01-20 19:22:44.962,Headless-Wan2GP,"[{'role': 'user', 'content': '# TASK: 2026-01-...",3567649,187
2997,hitlabstudios/dataclaw-peteromallet,f1066891-70d6-4fa4-95a1-00227cc9c973,claude-opus-4-6,main,2026-02-11 18:40:39.033,2026-02-11 20:08:56.499,Headless-Wan2GP,"[{'role': 'user', 'content': 'Install and run ...",60759044,12365
2998,hitlabstudios/dataclaw-peteromallet,2809ffed-5aec-46e0-a0d7-dbce58e6d9a5,claude-opus-4-5-20251101,main,2026-02-02 19:47:12.376,2026-02-03 11:57:00.313,banodoco-wrapped,"[{'role': 'user', 'content': 'See here, can yo...",47120023,1686
